# TP 10 — Un pipeline complet, de la variable au seuil### Module 6 — Apprentissage à l'échelle et gouvernance> **Filière Ingénierie Financière.** La filière Art Numérique traite le **TP11**> (vision par ordinateur distribuée) pendant que vous faites celui-ci.>> Le corrigé du TP11 vous sera distribué : **ses concepts sont au programme de> l'examen**, sa mise en œuvre ne l'est pas. Deux points à en retenir en> particulier — l'amortissement du chargement d'un modèle par `pandas_udf` à> itérateur, et le décalage de distribution.**Durée :** 2 heures · **Noté sur 20**---## Ce que vous devez savoir faire à la fin1. Construire des **variables sur fenêtres** avec Spark, à l'échelle.2. Assembler un `Pipeline` MLlib complet, sauvegardable.3. **Détecter une fuite de données** volontairement introduite dans ce TP.4. Choisir la bonne métrique sur des classes très déséquilibrées.5. **Choisir un seuil de décision à partir d'un coût métier**, pas d'une courbe.6. Décomposer les performances par sous-groupe et interpréter l'écart.## Barème| Exercice | Sujet | Points ||---|---|---|| 1 | Préparation et exploration | 2 || 2 | Construire des variables | 4 || 3 | Le pipeline, et la fuite | 5 || 4 | Évaluer : la bonne métrique | 4 || 5 | Le seuil, et son coût | 3 || 6 | Par sous-groupe | 2 |## Avertissement> Ce TP contient **une fuite de données volontairement introduite**. L'exercice 3 vous> demandera de la trouver. Si vous obtenez un score parfait, méfiez-vous : c'est le symptôme.

---# Exercice 1 — Préparation  *(2 points)*

In [ ]:
# 1.1 — Session Sparkfrom pyspark.sql import SparkSession, functions as F, Windowfrom pyspark.ml import Pipelinefrom pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssemblerfrom pyspark.ml.classification import GBTClassifier, LogisticRegressionfrom pyspark.ml.evaluation import BinaryClassificationEvaluatorimport timeUTILISATEUR = "etudiant"spark = (SparkSession.builder         .appName("TP10 - pipeline MLlib")         .master("local[4]")         .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020")         .config("spark.sql.shuffle.partitions", "16")         .getOrCreate())spark.sparkContext.setLogLevel("WARN")print("Spark", spark.version, "| UI :", spark.sparkContext.uiWebUrl)

In [ ]:
# 1.2 — Les transactions, en Parquet trié (produites au TP2)CHEMIN = f"hdfs://namenode:8020/user/{UTILISATEUR}/formats/pq_trie"brut = spark.read.parquet(CHEMIN)n = brut.count()n_fraude = brut.filter(F.col("est_fraude")).count() if "est_fraude" in brut.columns else 0print(f"{n:,} transactions")print(f"{n_fraude:,} fraudes  ({100*n_fraude/n:.3f} %)")brut.printSchema()

> **Si la colonne `est_fraude` est absente**, relancez le générateur du module 1 et> réécrivez le Parquet en conservant toutes les colonnes.### Q1 *(2 pts)* —- **a.** Quelle proportion de fraudes ? Quel taux d'exactitude obtiendrait un modèle qui  prédirait toujours « pas de fraude » ?- **b.** Que conclure sur l'usage de l'exactitude comme métrique ici ?

**Votre réponse :***(rédigez ici)*

---# Exercice 2 — Construire des variables  *(4 points)***Objectif.** C'est ici que Spark apporte réellement quelque chose : calculer des agrégats surfenêtres, sur des volumes qu'aucune machine seule ne traiterait.

In [ ]:
# 2.1 — Trois variables sur fenêtre glissante par compte# Fenetre : les 20 operations precedentes du meme compte, EXCLUANT la courante.w = (Window.partitionBy("id_compte")           .orderBy(F.col("horodatage").cast("long"))           .rowsBetween(-20, -1))enrichi = (brut    .withColumn("montant_moyen_compte", F.avg("montant").over(w))    .withColumn("nb_ops_precedentes",   F.count("*").over(w))    .withColumn("horodatage_precedent", F.lag("horodatage", 1).over(        Window.partitionBy("id_compte").orderBy("horodatage")))    .withColumn("delai_secondes",        F.col("horodatage").cast("long") - F.col("horodatage_precedent").cast("long"))    .withColumn("ecart_a_habitude",        F.when(F.col("montant_moyen_compte").isNull(), 0.0)         .otherwise(F.col("montant") / F.col("montant_moyen_compte"))))enrichi.select("id_compte", "horodatage", "montant", "montant_moyen_compte",               "nb_ops_precedentes", "delai_secondes", "ecart_a_habitude").show(8, False)

### Q2 *(4 pts)* —- **a.** La fenêtre est définie par `rowsBetween(-20, -1)`. Pourquoi **-1** et non **0** ?  Que se passerait-il avec `rowsBetween(-20, 0)` ?- **b.** Ouvrez la Spark UI. Combien d'`Exchange` cette cellule a-t-elle produits ? Pourquoi ?- **c.** La colonne `id_compte` suit une loi de puissance (module 4). Quel risque cela  fait-il courir à ce calcul ? Comment le vérifieriez-vous ?- **d.** `ecart_a_habitude` vaut 0 quand la moyenne est nulle — c'est-à-dire pour les  premières opérations d'un compte. Est-ce un bon choix ? Proposez une alternative.

**Votre réponse :***(rédigez ici)*

---# Exercice 3 — Le pipeline, et la fuite  *(5 points)*

In [ ]:
# 3.1 — Un pipeline "naïf" — attention à son contenuCATEG = ["pays_transaction", "canal", "devise", "statut"]NUM   = ["montant", "montant_moyen_compte", "nb_ops_precedentes",         "ecart_a_habitude"]donnees = (enrichi    .fillna({"delai_secondes": 0, "montant_moyen_compte": 0.0, "nb_ops_precedentes": 0})    .withColumn("label", F.col("est_fraude").cast("double"))    .filter(F.col("montant").isNotNull()))entrainement, test = donnees.randomSplit([0.8, 0.2], seed=42)print(f"entrainement : {entrainement.count():,}")print(f"test         : {test.count():,}")

In [ ]:
# 3.2 — Construire et ajuster le pipelineetapes = [    StringIndexer(inputCols=CATEG,                  outputCols=[c + "_i" for c in CATEG],                  handleInvalid="keep"),    OneHotEncoder(inputCols=[c + "_i" for c in CATEG],                  outputCols=[c + "_v" for c in CATEG]),    VectorAssembler(inputCols=NUM + [c + "_v" for c in CATEG],                    outputCol="features"),    GBTClassifier(featuresCol="features", labelCol="label", maxIter=20, maxDepth=5),]debut = time.time()modele = Pipeline(stages=etapes).fit(entrainement)print(f"entraine en {time.time()-debut:.1f} s")predictions = modele.transform(test).cache()auc_roc = BinaryClassificationEvaluator(labelCol="label",              metricName="areaUnderROC").evaluate(predictions)auc_pr  = BinaryClassificationEvaluator(labelCol="label",              metricName="areaUnderPR").evaluate(predictions)print(f"AUC-ROC : {auc_roc:.4f}")print(f"AUC-PR  : {auc_pr:.4f}")

### Q3a *(2 pts)* — Les scores obtenus vous paraissent-ils crédibles pour un problème dedétection de fraude ? Qu'est-ce qui devrait vous alerter ?

**Votre réponse :***(rédigez ici)*

In [ ]:
# 3.3 — Quelles variables le modèle utilise-t-il vraiment ?gbt = modele.stages[-1]noms = NUM + [f"{c}_v" for c in CATEG]importances = gbt.featureImportances# On agrege les importances par variable d'origineassembleur = modele.stages[-2]print("Importances des variables numeriques :")for i, nom in enumerate(NUM):    print(f"  {nom:<26} {importances[i]:.4f}")print("\n(les variables categorielles sont eclatees en plusieurs colonnes)")print("\nSomme des importances des colonnes categorielles :",      round(sum(importances[len(NUM):]), 4))

In [ ]:
# 3.4 — Examiner la variable suspectedonnees.groupBy("statut", "est_fraude").count().orderBy("statut", "est_fraude").show()

### Q3b *(3 pts)* —- **a.** Quelle variable est à l'origine de la fuite ? Comment le tableau de 3.4 le montre-t-il ?- **b.** De quel **type** de fuite s'agit-il ? Justifiez en vous demandant à quel moment cette  variable prend sa valeur.- **c.** Corrigez le pipeline, réentraînez, et comparez les scores. Commentez l'écart.

In [ ]:
# 3.5 — À VOUS : corriger et réentraînerCATEG_CORRIGE = ...   # retirez la variable fautive# ... reconstruisez le pipeline et mesurez a nouveau

**Votre réponse :***(rédigez ici)*

---# Exercice 4 — Évaluer : la bonne métrique  *(4 points)*

In [ ]:
# 4.1 — La matrice de confusion au seuil par défaut de 0,5def matrice(predictions, seuil=0.5):    p = predictions.withColumn("pred",        (F.element_at(F.col("probability").cast("array<double>"), 2) >= seuil).cast("double"))    vp = p.filter((F.col("label") == 1) & (F.col("pred") == 1)).count()    fn = p.filter((F.col("label") == 1) & (F.col("pred") == 0)).count()    fp = p.filter((F.col("label") == 0) & (F.col("pred") == 1)).count()    vn = p.filter((F.col("label") == 0) & (F.col("pred") == 0)).count()    return vp, fn, fp, vnvp, fn, fp, vn = matrice(pred2, 0.5)total = vp + fn + fp + vnprint(f"vrais positifs (fraudes detectees)  : {vp:>7,}")print(f"faux negatifs  (fraudes manquees)   : {fn:>7,}")print(f"faux positifs  (fausses alertes)    : {fp:>7,}")print(f"vrais negatifs                      : {vn:>7,}")print(f"\nexactitude : {(vp+vn)/total:.4f}")print(f"precision  : {vp/(vp+fp) if vp+fp else 0:.4f}")print(f"rappel     : {vp/(vp+fn) if vp+fn else 0:.4f}")

In [ ]:
# 4.2 — Comparer avec le modèle constantconstant_exactitude = vn_total = (pred2.filter(F.col("label") == 0).count()                                  / pred2.count())print(f"exactitude du modele constant 'jamais de fraude' : {constant_exactitude:.4f}")print(f"exactitude de notre modele                       : {(vp+vn)/total:.4f}")print(f"\nrappel du modele constant : 0.0000")print(f"rappel de notre modele    : {vp/(vp+fn) if vp+fn else 0:.4f}")

### Q4 *(4 pts)* —- **a.** Comparez l'exactitude de votre modèle à celle du modèle constant. L'écart est-il  impressionnant ?- **b.** Comparez maintenant les rappels. Que conclure ?- **c.** Précision et rappel : lequel privilégieriez-vous pour la détection de fraude, et  pourquoi ? Y a-t-il une réponse unique ?- **d.** Pourquoi l'AUC-PR est-elle plus informative que l'AUC-ROC sur ce jeu de données ?

**Votre réponse :***(rédigez ici)*

---# Exercice 5 — Le seuil, et son coût  *(3 points)***Objectif.** Le point qui distingue un ingénieur d'un étudiant : le seuil de décision sechoisit sur un **coût métier**, pas sur une courbe.

In [ ]:
# 5.1 — Les coûts métierCOUT_FRAUDE_MANQUEE = 2000    # EUR : montant moyen non recupereCOUT_FAUSSE_ALERTE  = 15      # EUR : traitement + mecontentement clientresultats = []for seuil in [0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]:    vp, fn, fp, vn = matrice(pred2, seuil)    cout = fn * COUT_FRAUDE_MANQUEE + fp * COUT_FAUSSE_ALERTE    resultats.append((seuil, vp, fn, fp, cout))print(f"{'seuil':>6} {'detectees':>10} {'manquees':>10} {'f.alertes':>11} {'cout (EUR)':>13}")print("-" * 54)for s, vp, fn, fp, cout in resultats:    print(f"{s:>6.1f} {vp:>10,} {fn:>10,} {fp:>11,} {cout:>13,.0f}")meilleur = min(resultats, key=lambda r: r[4])print(f"\nseuil optimal : {meilleur[0]}  ->  {meilleur[4]:,.0f} EUR")

In [ ]:
# 5.2 — Et si les coûts changeaient ?print(f"{'f.negatif':>10} {'f.positif':>10} {'seuil optimal':>15} {'cout':>13}")print("-" * 52)for cfn, cfp in [(2000, 15), (2000, 100), (500, 15), (10000, 15), (2000, 5)]:    best = min(((s, fn*cfn + fp*cfp) for s, vp, fn, fp, _ in resultats),               key=lambda x: x[1])    print(f"{cfn:>10} {cfp:>10} {best[0]:>15.1f} {best[1]:>13,.0f}")

### Q5 *(3 pts)* —- **a.** Quel seuil minimise le coût ? Est-il égal à 0,5 ?- **b.** Le coût n'est pas monotone : il décroît puis remonte. Expliquez pourquoi.- **c.** Dans le tableau 5.2, comment le seuil optimal évolue-t-il quand le coût d'une fausse  alerte augmente ? Et quand celui d'une fraude manquée augmente ? Cela vous paraît-il logique ?- **d.** Le service client indique qu'il ne peut pas traiter plus de 2 000 alertes par jour.  Comment intégrez-vous cette contrainte ?

**Votre réponse :***(rédigez ici)*

---# Exercice 6 — Par sous-groupe  *(2 points)*

In [ ]:
# 6.1 — Décomposer les performances par paysSEUIL = meilleur[0]par_pays = []for r in pred2.select("pays_transaction").distinct().collect():    p = r.pays_transaction    sous = pred2.filter(F.col("pays_transaction") == p)    n_sous = sous.count()    if n_sous < 500:        continue    try:        auc = BinaryClassificationEvaluator(labelCol="label",                  metricName="areaUnderROC").evaluate(sous)    except Exception:        auc = float("nan")    vp, fn, fp, vn = matrice(sous, SEUIL)    rappel = vp / (vp + fn) if (vp + fn) else float("nan")    par_pays.append((p, n_sous, vp + fn, auc, rappel))print(f"{'pays':>6} {'lignes':>9} {'fraudes':>9} {'AUC-ROC':>9} {'rappel':>9}")print("-" * 46)for p, n_s, nf, auc, rap in sorted(par_pays, key=lambda x: -x[1]):    print(f"{p:>6} {n_s:>9,} {nf:>9,} {auc:>9.3f} {rap:>9.3f}")

### Q6 *(2 pts)* —- **a.** Les performances sont-elles homogènes entre pays ? Relevez l'écart entre le meilleur  et le pire.- **b.** Un écart peut avoir deux causes très différentes. Lesquelles ? Comment les  distingueriez-vous ?- **c.** Que feriez-vous de ce constat ? Distinguez ce qui relève de vous et ce qui relève  d'une décision métier.

**Votre réponse :***(rédigez ici)*

---# Sauvegarder le modèle

In [ ]:
# Sauvegarder le PIPELINE entier, pas seulement le modèleCHEMIN_MODELE = f"hdfs://namenode:8020/user/{UTILISATEUR}/modeles/fraude_v1"modele2.write().overwrite().save(CHEMIN_MODELE)# Verifier qu'on peut le recharger et l'utiliser tel quelfrom pyspark.ml import PipelineModelrecharge = PipelineModel.load(CHEMIN_MODELE)print("modele recharge, etapes :", [type(s).__name__ for s in recharge.stages])print("verification :", recharge.transform(test.limit(100)).count(), "lignes predites")

### Question finale *(non notée, mais attendue au projet)*Pourquoi sauvegarde-t-on le **pipeline entier** et non le seul `GBTClassifier` ?Que devrait faire l'équipe de production si vous ne lui transmettiez que le classifieur ?

**Votre réponse :***(rédigez ici)*

---# Synthèse| Observation | Votre chiffre | Ce que vous en concluez ||---|---|---|| Proportion de fraudes | | || AUC-ROC avec la fuite | | || AUC-ROC sans la fuite | | || Exactitude du modèle constant | | || Seuil optimal, et son coût | | || Écart d'AUC entre pays | | |**Question de conclusion.** Parmi tout ce que vous avez fait aujourd'hui, quelle étape auraitle plus d'impact si elle était omise en production ?

In [ ]:
pred2.unpersist()spark.stop()print("Session fermee.")

---## Avant de rendre- [ ] Les questions **Q1 à Q6** sont rédigées et justifiées.- [ ] Vous avez **identifié et corrigé** la fuite de données (Q3b).- [ ] Les scores avant et après correction sont reportés, avec votre commentaire.- [ ] Le tableau des coûts par seuil figure dans vos réponses.- [ ] Le tableau de synthèse est complété.- [ ] Notebook exporté en HTML et déposé.**Bon TP — et bonne fin de semestre.**